In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.1 Queries, keys and values

Attention is a soft, content-based lookup. Each token emits a **query** (what
am I looking for), a **key** (what do I offer), and a **value** (what I pass on).
Token i attends to token j in proportion to query_i . key_j.

In [ ]:
from torch import nn
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, dim, hs = 3, 8, 8
x = torch.randn(tokens, dim)
w_q = nn.Linear(dim, hs, bias=False)
w_k = nn.Linear(dim, hs, bias=False)
w_v = nn.Linear(dim, hs, bias=False)
q, k, v = w_q(x), w_k(x), w_v(x)
out, weights = scaled_dot_product_attention(
    q.unsqueeze(0), k.unsqueeze(0), v.unsqueeze(0))
print("attention weights (each row sums to 1):")
print(weights[0].detach().round(decimals=2))

Each row is one token's distribution of attention over all tokens. The output
is the weighted sum of values, a blend of the information the token chose.

In [ ]:
assert torch.allclose(weights.sum(-1), torch.ones(1, tokens), atol=1e-6)